In [1]:
import os
import json

In [2]:
p = "/shared_workspace_mfs/yanruo/repo/Agent/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v3-distilling"

In [ ]:

from collections import defaultdict
from tqdm import tqdm
from pathlib import Path

def parse_messages(instance_dir, parsed_result):
    if not os.path.exists(instance_dir):
        return False
    need_trajs = []
    ori_instances = sorted(os.listdir(instance_dir), key=lambda x: float(x.replace(".json", "").split("-")[-1]), reverse=True)
    for fname in ori_instances:
        need_trajs.append(fname)
        if "-1-" in fname:
            break
    parsed_result["have_muilt_original_sources_file"] = len(need_trajs) != len(ori_instances)
    messages = []
    idx = 0
    have_all_traj_saved = True
    for i, fname in enumerate(reversed(need_trajs)):
        fpath = os.path.join(instance_dir, fname)
        i_step = int(fname.split("-")[1])
        jd = json.load(open(fpath))

        if i == 0:
            messages += jd["messages"]
            idx += len(jd["messages"])
        else:
            messages += jd["messages"][idx+1:]
            idx += len(jd["messages"][idx+1:]) + 1

        resp = jd["response"]["choices"][0]["message"]
        resp["step"] = i
        resp["usage"] = jd["response"]["usage"]
        resp["usage"].pop("completion_tokens_details")
        resp["usage"].pop("prompt_tokens_details")
        # resp.pop("tool_calls")
        # resp.pop("function_call")
        resp["source_file"] = fpath
        messages.append(resp)
        if i + 1 != i_step:
            have_all_traj_saved= False

    parsed_result["have_all_traj_saved"] = have_all_traj_saved
    parsed_result["messages"] = messages
    return True

def parse_messages_with_toolcalls(instance_dir, parsed_result):
    if not os.path.exists(instance_dir):
        return False
    need_trajs = []
    ori_instances = sorted(os.listdir(instance_dir), key=lambda x: float(x.replace(".json", "").split("-")[-1]), reverse=True)
    for fname in ori_instances:
        need_trajs.append(fname)
        if "-1-" in fname:
            break
    parsed_result["have_muilt_original_sources_file"] = len(need_trajs) != len(ori_instances)
    messages = []
    idx = 0
    have_all_traj_saved = True
    for i, fname in enumerate(reversed(need_trajs)):
        fpath = os.path.join(instance_dir, fname)
        i_step = int(fname.split("-")[1])
        jd = json.load(open(fpath))

        if i == 0:
            messages += jd["messages"]
            idx += len(jd["messages"])
        else:
            messages += jd["messages"][idx+1:]
            idx += len(jd["messages"][idx+1:]) + 1

        resp = jd["response"]["choices"][0]["message"]
        resp["step"] = i
        resp["usage"] = jd["response"]["usage"]
        resp["usage"].pop("completion_tokens_details")
        resp["usage"].pop("prompt_tokens_details")
        # resp.pop("tool_calls")
        resp.pop("function_call")
        resp["source_file"] = fpath
        messages.append(resp)
        if i + 1 != i_step:
            have_all_traj_saved= False

    parsed_result["have_all_traj_saved"] = have_all_traj_saved
    parsed_result["messages"] = messages
    parsed_result["tools"] = resp["kwargs"]["tools"]
    return True

def parse_error(instance_infer_dir, parsed_result):
    pred_jsonl = os.path.join(instance_infer_dir, 'pred.jsonl')
    if not os.path.exists(pred_jsonl):
        return False
    jd = json.loads(open(pred_jsonl, 'r').readline())
    if "error" in jd:
        parsed_result["error"] = jd["error"]
    return True

def parse_eval_result(instance_eval_dir, parsed_result, instance_id):
    report_file = os.path.join(instance_eval_dir, "report.json")
    instance_id = os.path.basename(instance_eval_dir)
    if not os.path.exists(report_file):
        return False
    jd = json.load(open(report_file))
    if instance_id not in jd:
        return False
    jd = jd[instance_id]
    parsed_result.update({
        "patch_is_None": jd["patch_is_None"],
        "patch_exists": jd["patch_exists"],
        "patch_successfully_applied": jd["patch_successfully_applied"],
        "resolved": jd["resolved"],
    })
    return True

def parse_and_save_instance_trajectory(output_dir, instance_id, save=True):
    parsed_result = {
        "instance_id": instance_id,
        "repo": instance_id.split("__")[0],
    }
    parsed_error = []

    # if not parse_error(os.path.join(output_dir, "instances", instance_id), parsed_result):
    #     parsed_error.append("parse_pred_error")
    #     return parsed_error
    # if not parse_eval_result(os.path.join(output_dir, "instances", instance_id), parsed_result, instance_id):
    #     pass

    if not parse_messages(os.path.join(output_dir, "llm_completions", instance_id), parsed_result):
        parsed_error.append("parse_messages_error")

    if parsed_result.get("have_all_traj_saved") is False:
        parsed_error.append("not_all_traj_saved")

    if save:
        save_path = os.path.join(output_dir, "instances", instance_id, "parsed_traj.json")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        with open(save_path, "w") as f:
            json.dump(parsed_result, f, indent=2)
        # print(f"Saved parsed trajectory to {instance_id}")
    return parsed_error

def parse_all_trajectories(output_dir):
    parse_errors = defaultdict(list)
    for instance_id in tqdm(os.listdir(os.path.join(output_dir, "instances"))):
        parsed_error_list = parse_and_save_instance_trajectory(output_dir, instance_id)
        if parsed_error_list:
            for err in parsed_error_list:
                parse_errors[err].append(instance_id)

    print(parse_errors)
    return parse_errors

# output_dir = "/shared_workspace/yanruo/github/OpenHands/evaluation/evaluation_outputs/outputs/SWE-bench-Live__SWE-bench-Live-lite/CodeActAgent/deepswe32b_maxiter_50_N_distilling"
output_dir = "/shared_workspace_mfs/yanruo/repo/Agent/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v3-distilling"
# instance_id = "aws-cloudformation__cfn-lint-3770"
# parse_and_save_instance_trajectory(output_dir, instance_id, save=False)
parsed_data = parse_all_trajectories(output_dir)



100%|██████████| 1563/1563 [00:19<00:00, 78.33it/s] 

defaultdict(<class 'list'>, {'not_all_traj_saved': ['pydata__xarray-10185', 'bridgecrewio__checkov-6933', 'yt-dlp__yt-dlp-12684', 'stanfordnlp__dspy-779', 'pvlib__pvlib-python-2055', 'keras-team__keras-19773', 'pdm-project__pdm-2817', 'sphinx-doc__sphinx-12516', 'huggingface__smolagents-285', 'beeware__briefcase-2241', 'koxudaxi__datamodel-code-generator-2389', 'matplotlib__matplotlib-28104', 'conan-io__conan-17366', 'pypa__twine-1106', 'fonttools__fonttools-3559', 'pdm-project__pdm-3170', 'keras-team__keras-19564', 'falconry__falcon-2459', 'aws-cloudformation__cfn-lint-4002', 'pylint-dev__pylint-9772', 'geopandas__geopandas-3271', 'aiogram__aiogram-1670', 'reflex-dev__reflex-4720', 'instructlab__instructlab-367', 'matplotlib__matplotlib-29388', 'python-control__python-control-1145', 'geopandas__geopandas-3214', 'sphinx-doc__sphinx-12495', 'python-control__python-control-1096', 'conan-io__conan-15737', 'instructlab__instructlab-1413', 'beeware__briefcase-2035', 'sphinx-doc__sphinx-1259

In [107]:
from copy import deepcopy
def transer_resoning_content_back(messages):
    for msg in messages:
        if msg["role"] != "assistant":
            continue
        content = msg["content"]
        reasoning_content = msg["reasoning_content"]
        if reasoning_content is None:
            # print(json.dumps(msg, indent=2))
            continue
        # print(reasoning_content)
        content = f"<think>{reasoning_content}</think>{content}"
        msg["content"] = content
        msg.pop("reasoning_content")
        msg.pop("usage")
        msg.pop("source_file")
        msg.pop("step")
    return messages

def generate_subsequence(output_dir):
    traj_dir = os.path.join(output_dir, "parsed_trajectories")
    subseq_dir = os.path.join(output_dir, "parsed_trajectories_subseq")
    os.system(f"rm -r {subseq_dir}")
    os.makedirs(subseq_dir, exist_ok=True)
    cnt = defaultdict(int)
    for instance_id in os.listdir(traj_dir):
        fpath = os.path.join(traj_dir, instance_id)
        jd = json.load(open(fpath))
        if jd.get("resolved", False) is False:
            continue
        if jd.get("error", "x") is not None:
            error = jd.get("error", "x")
            cnt[error] += 1
            continue
        cnt["resolved"] += 1
        ori_message = deepcopy(jd["messages"])
        message = transer_resoning_content_back(jd["messages"])

        start = 0
        end = 0
        msg_len = len(message)
        while end < len(message):
            if message[end]["role"] != "assistant":
                end += 1
                continue
            # success = not check_error(message[end+1]["content"])
            with open(os.path.join(subseq_dir, f"{instance_id.replace(".json", "")}_{msg_len}_{start}_{end}.json"), 'w') as f:
                json.dump({
                    "messages": message[start:end+1],
                    "source_file": fpath,
                    "step": ori_message[end]["step"] + 1
                }, f, indent=4)
            end += 1
    print(len(os.listdir(subseq_dir)))
    print(cnt)

generate_subsequence(output_dir)

273
defaultdict(<class 'int'>, {'RuntimeError: Agent reached maximum iteration. Current iteration: 100, max iteration: 100': 1, 'resolved': 24})


In [ ]:
def generate_from_old_format(output_dir):
    output_jsonl = os.path.join(output_dir, "output.jsonl")
    for line in open(output_jsonl):
        jd = json.loads(line)
        instance_id = jd["instance_id"]
        instance_dir = os.path.join(output_dir, "instances", instance_id)
        os.makedirs(instance_dir, exist_ok=True)
        with open(os.path.join(instance_dir, "pred.jsonl"), "w") as f:
            json.dump(jd, f)
    return
output_dir = "/shared_workspace/yanruo/github/OpenHands/evaluation/evaluation_outputs/outputs/SWE-bench-Live__SWE-bench-Live-lite/CodeActAgent/DeepSWE32B_maxiter_100_N_v0.50.0-no-hint-run_1"
generate_from_old_format(output_dir)

In [ ]:
import os
def copy_from_old(output_dir):
    instance_dir = os.path.join(output_dir, "instances")
    for instance_id in os.listdir(instance_dir):
        eval_instance_id = os.path.join(output_dir, "eval_outputs", instance_id)
        infer_instance_id = os.path.join(output_dir, "instances", instance_id)
        if os.path.exists(eval_instance_id):
            os.system(f"cp {eval_instance_id}/* {infer_instance_id}")
output_dir = "/shared_workspace/yanruo/github/OpenHands/evaluation/evaluation_outputs/outputs/SWE-bench-Live__SWE-bench-Live-full/CodeActAgent/DeepSWE32B_maxiter_50_N_v2-distilling"


copy_from_old(output_dir)

In [3]:
from collections import defaultdict
from pathlib import Path
import os
import json
from tqdm import tqdm

def check_status(output_dir):
    cnt = defaultdict(int)
    output_dir = Path(output_dir)
    instance_dir = output_dir / "instances"
    for instance_id in tqdm(os.listdir(instance_dir)):
        pred = output_dir / "instances" / instance_id / "pred.jsonl"
        if not pred.exists():
            continue
        cnt["finished_inference"] += 1
        jd = json.load(open(pred))
        cnt["non_empty_patch"] += 1 if jd["test_result"]["git_patch"] else 0
        if jd["error"]:
            cnt[jd["error"]] += 1

        eval_log = output_dir / "instances" / instance_id / "run_instance.log"
        if not eval_log.exists():
            continue
        cnt["have_evaluation"] += 1
        report_json = output_dir / "instances" / instance_id / "report.json"
        if not report_json.exists():
            continue
        cnt["finished_evaluation"] += 1
        try:
            rjd = json.load(open(report_json))
            cnt["resolved"] += 1 if rjd[instance_id]["resolved"] else 0
        except:
            print(instance_id)

    print(json.dumps(cnt, indent = 2))



In [21]:
# output_dir = "/shared_workspace/yanruo/github/OpenHands/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/deepswe32b_maxiter_50_N_v2-distilling"
# check_status(output_dir)

output_dir = "/shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/DeepSWE32B_maxiter_50_N_v3-distilling"
check_status(output_dir)

100%|██████████| 33/33 [00:00<00:00, 665.31it/s]

{
  "finished_inference": 33,
  "non_empty_patch": 32,
  "have_evaluation": 32,
  "finished_evaluation": 20,
  "resolved": 2,
  "RuntimeError: Agent reached maximum iteration. Current iteration: 50, max iteration: 50": 2,
  "AgentStuckInLoopError: Agent got stuck in a loop": 1
}


In [5]:

common_dir = "/shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs"
for dataset in os.listdir(common_dir):
    if "princeton" in dataset:
        continue
    dataset_dir = os.path.join(common_dir, dataset,"CodeActAgent")
    for datacollect in os.listdir(dataset_dir):
        datacollected_dir = os.path.join(dataset_dir, datacollect)
        print("========================================")
        print(dataset, datacollect)
        check_status(datacollected_dir)
        print()



SWE-bench-Live__SWE-bench-Live-verified DeepSWE32B_maxiter_50_N_distilling


100%|██████████| 499/499 [00:01<00:00, 454.02it/s]


{
  "finished_inference": 499,
  "non_empty_patch": 452,
  "RuntimeError: Agent reached maximum iteration. Current iteration: 50, max iteration: 50": 62,
  "have_evaluation": 452,
  "finished_evaluation": 318,
  "resolved": 33,
  "AgentStuckInLoopError: Agent got stuck in a loop": 65
}

SWE-bench-Live__SWE-bench-Live-full DeepSWE32B_maxiter_50_N_distilling


100%|██████████| 100/100 [00:00<00:00, 597.78it/s]


{
  "finished_inference": 100,
  "non_empty_patch": 92,
  "RuntimeError: Agent reached maximum iteration. Current iteration: 50, max iteration: 50": 11,
  "AgentStuckInLoopError: Agent got stuck in a loop": 15
}

SWE-bench-Live__SWE-bench-Live-full Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v2-distilling


...

{
  "finished_inference": 11,
  "non_empty_patch": 11,
  "have_evaluation": 11,
  "finished_evaluation": 1,
  "resolved": 0
}

SWE-bench-Rebench-test deepswe32b_maxiter_50_N_v2-distilling


100%|██████████| 3214/3214 [00:06<00:00, 533.24it/s]


{
  "finished_inference": 3214,
  "non_empty_patch": 2979,
  "AgentStuckInLoopError: Agent got stuck in a loop": 354,
  "have_evaluation": 1988,
  "finished_evaluation": 1251,
  "resolved": 206,
  "RuntimeError: Agent reached maximum iteration. Current iteration: 50, max iteration: 50": 232,
  "STATUS$ERROR_LLM_INTERNAL_SERVER_ERROR": 87
}

SWE-bench-Rebench-test DeepSWE32B_maxiter_50_N_v2-distilling


100%|██████████| 3214/3214 [00:05<00:00, 558.46it/s]

{
  "finished_inference": 3214,
  "non_empty_patch": 2979,
  "AgentStuckInLoopError: Agent got stuck in a loop": 354,
  "have_evaluation": 1988,
  "finished_evaluation": 1251,
  "resolved": 206,
  "RuntimeError: Agent reached maximum iteration. Current iteration: 50, max iteration: 50": 232,
  "STATUS$ERROR_LLM_INTERNAL_SERVER_ERROR": 87
}



In [ ]:
def clean_eval(dir):
    dir = Path(dir)
    report_json = json.load(open(dir / "report.json"))
    for instance_id, status in report_json["instance_ids"].items():
        status, error = status
        if status in ["finished"]:
            file = f'{dir / "instances" / instance_id / "pred.jsonl"} '
            os.system(f"rm {file}")
        if error in ["Error: STATUS$ERROR_LLM_INTERNAL_SERVER_ERROR"]:
            file = f'{dir / "instances" / instance_id / "run_instance.log"} '
            file += f'{dir / "instances" / instance_id / "patch.diff"} '
            file += f'{dir / "instances" / instance_id / "report.json"} '
            file += f'{dir / "instances" / instance_id / "pred.jsonl"} '
            print(file)
            os.system(f"rm {file}")

dir = "/shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Live__SWE-bench-Live-full/CodeActAgent/DeepSWE32B_maxiter_50_N_v3-distilling-r5"
clean_eval(dir)

/shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v2-distilling/instances/zarr-developers__zarr-python-2851/run_instance.log /shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v2-distilling/instances/zarr-developers__zarr-python-2851/patch.diff /shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v2-distilling/instances/zarr-developers__zarr-python-2851/report.json 


rm: cannot remove '/shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v2-distilling/instances/zarr-developers__zarr-python-2851/run_instance.log': No such file or directory
rm: cannot remove '/shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v2-distilling/instances/zarr-developers__zarr-python-2851/patch.diff': No such file or directory
rm: cannot remove '/shared_workspace/yanruo/repo/OpenHands-Data-Collect/evaluation/evaluation_outputs/outputs/SWE-bench-Rebench-test/CodeActAgent/Qwen3-Coder-30B-A3B-Instruct_maxiter_100_N_v2-distilling/instances/zarr-developers__zarr-python-2851/report.json': No such file or directory
